In [2]:
import torch

history = {}

def store_version(entity, value):
    if entity not in history:
        history[entity] = []

    # TODO 1: Append a copy of value
    history[entity].append(value.clone())


def read_version(entity, version):
    # TODO 2: Return a copy of selected version
    return history[entity][version].clone()


v1 = torch.tensor([1., 2., 3.])
v2 = torch.tensor([1., 2., 3., 4., 5.])
v3 = torch.tensor([1., 2., 3., 4., 5., 6., 7., 8.])

store_version("dragon", v1)
store_version("dragon", v2)
store_version("dragon", v3)

for i in range(3):
    result = read_version("dragon", i)
    print(f"Version {i+1}: shape={result.shape}")

Version 1: shape=torch.Size([3])
Version 2: shape=torch.Size([5])
Version 3: shape=torch.Size([8])


In [3]:
import torch

versions = history["dragon"]

# Record original lengths
lengths = torch.tensor(
    [v.numel() for v in versions],
    dtype=torch.int64
)

max_dim = int(lengths.max().item())

# Allocate a padded matrix
padded = torch.zeros(
    len(versions),
    max_dim
)

# Copy each variable-length vector
for i, v in enumerate(versions):

    # TODO 1:
    # Copy v into the first v.numel()
    # positions of row i
    padded[i, :v.numel()] = v.clone()

print("Padded matrix:\n", padded)
print("Shape:", padded.shape)
print("Original lengths:", lengths)

Padded matrix:
 tensor([[1., 2., 3., 0., 0., 0., 0., 0.],
        [1., 2., 3., 4., 5., 0., 0., 0.],
        [1., 2., 3., 4., 5., 6., 7., 8.]])
Shape: torch.Size([3, 8])
Original lengths: tensor([3, 5, 8])


In [6]:
def read_padded(index):

    # TODO 2:
    # Recover the original length
    n = lengths[index].item()

    # Return only the original elements
    return padded[index, :n].clone()


for i, original in enumerate(versions):
    recovered = read_padded(i)

    print(
        f"Version {i+1}:",
        recovered.shape,
        "Correct:",
        torch.equal(recovered, original)
    )

Version 1: torch.Size([3]) Correct: True
Version 2: torch.Size([5]) Correct: True
Version 3: torch.Size([8]) Correct: True


In [7]:
def tensor_bytes(t):
    return t.numel() * t.element_size()

dynamic_bytes = sum(
    tensor_bytes(v) for v in versions
)

padded_bytes = tensor_bytes(padded)
metadata_bytes = tensor_bytes(lengths)

print("Dynamic payload:", dynamic_bytes, "bytes")
print("Padded payload:", padded_bytes, "bytes")
print("Length metadata:", metadata_bytes, "bytes")
print("Padded + metadata:", padded_bytes + metadata_bytes)

Dynamic payload: 64 bytes
Padded payload: 96 bytes
Length metadata: 24 bytes
Padded + metadata: 120
